In [16]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

In [17]:
df = pd.read_excel('who_aap_2021_v9_11august2022.xlsx', sheet_name='AAP_2022_city_v9')

In [18]:
df['location_str'] = df['City or Locality'] + ', ' + df['WHO Country Name']

# Unikalne lokalizacje
unique_locations = df['location_str'].unique()

In [19]:
# Słownik do przechowywania wyników
location_to_coords = {}

In [20]:
# Inicjalizacja geolokatora Nominatim
geolocator = Nominatim(user_agent="my_geocoder_app")

In [21]:
# Iteracja po unikalnych lokalizacjach
for loc in unique_locations:
    try:
        location = geolocator.geocode(loc, timeout=10)
        if location:
            location_to_coords[loc] = (location.latitude, location.longitude)
        else:
            location_to_coords[loc] = (None, None)
    except Exception as e:
        print(f"Błąd przy przetwarzaniu '{loc}': {e}")
        location_to_coords[loc] = (None, None)
    time.sleep(1)  # zgodnie z zasadami Nominatim: max 1 zapytanie na sekundę

In [22]:
# Konwersja słownika na DataFrame
coords_df = pd.DataFrame.from_dict(location_to_coords, orient='index', columns=['Latitude', 'Longitude'])
coords_df.reset_index(inplace=True)
coords_df.rename(columns={'index': 'location_str'}, inplace=True)

In [23]:
# Połączenie współrzędnych z oryginalnym DataFrame
df = df.merge(coords_df, on='location_str', how='left')

In [24]:
# Usunięcie pomocniczej kolumny
df.drop(columns=['location_str'], inplace=True)

In [26]:
df.to_csv('who_with_coords.csv', index=False)